In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


import os
import string
import optuna
import gc

import sklearn
import sentence_transformers


from sentence_transformers import SentenceTransformer

from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

import xgboost as xgb
from xgboost import XGBClassifier
import catboost as catb
from catboost import CatBoostClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

2025-08-06 06:16:02.388021: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754460962.624522      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754460962.693535      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
RANDOM_STATE = 42


train_idx_path = "/kaggle/input/fake-or-real-the-impostor-hunt/data/train.csv"
train_path = "/kaggle/input/fake-or-real-the-impostor-hunt/data/train"
test_path = "/kaggle/input/fake-or-real-the-impostor-hunt/data/test"

In [3]:
def read_data(data_path: str) -> pd.DataFrame:
    data = []
    for dir in os.listdir(data_path):
        dir_path = os.path.join(data_path, dir)
        file_1_path = os.path.join(dir_path, "file_1.txt")
        file_2_path = os.path.join(dir_path, "file_2.txt")
        
        try:
            with open(file_1_path, "r", encoding="utf-8") as f:
                text1 = f.read().strip()
            with open(file_2_path, "r", encoding="utf-8") as f:
                text2 = f.read().strip()
            
            idx = int(dir_path[-4:])
            data.append((idx, text1, text2))        
        except Exception as e:
            print(f"Error reading directory {dir}: {e}")
        
    df = pd.DataFrame(data, columns=["id", "file1", "file2"])
    df = df.sort_values("id").reset_index(drop=True)
    
    return df


def clean_text(text: str) -> str:
    clean_punc = str.maketrans("", "", string.punctuation + "\n\t\r")
    clean_digit = str.maketrans("", "", string.digits)
    
    cleaned_text = text.translate(clean_punc)
    cleaned_text = cleaned_text.translate(clean_digit)
    cleaned_text = cleaned_text.lower()
    
    return cleaned_text


def clean_data(data: pd.DataFrame, cols: list) -> pd.DataFrame:
    df = data.copy()
    
    for col in cols:
        df[col] = df[col].apply(lambda row: clean_text(row))
    
    return df
    
    
def prepare_data(data: pd.DataFrame) -> pd.DataFrame:
    
    real_text = []
    fake_text = []
    
    for _, row in data.iterrows():
        if row["real_text_id"] == 1:
            real_text.append(row["file1"])
            fake_text.append(row["file2"])
        else:
            real_text.append(row["file2"])
            fake_text.append(row["file1"])
    
    real_df = pd.DataFrame({"text": real_text, "label": 1})
    fake_df = pd.DataFrame({"text": fake_text, "label": 0})
    
    df = pd.concat([real_df, fake_df]).sample(frac=1, random_state=RANDOM_STATE, ignore_index=True)
    
    return df


def get_embeddings(texts: np.ndarray, embedding_model) -> np.ndarray:
    embeddings = embedding_model.encode(texts,
                                        batch_size=64,
                                        show_progress_bar=True,
                                        convert_to_numpy=True,
                                        normalize_embeddings=True)
    return embeddings

def cross_val_skf(model,
                 X_train: np.ndarray,
                 y_train: np.ndarray,
                 n_splits: int=5,
                 shuffle: bool=True,
                 random_state: int=None) -> list:
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    scores = []
    
    for i, (tr_idx, vl_idx) in enumerate(skf.split(X_train, y_train), 1):
        X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
        y_tr, y_vl = y_train[tr_idx], y_train[vl_idx]
        
        if isinstance(model, XGBClassifier):
            model.fit(X_tr,
                  y_tr,
                  eval_set=[(X_vl, y_vl)],
                  verbose=0)
        
            best_iter = model.best_iteration + 1
            y_pred = model.predict(X_vl, iteration_range=(0, best_iter))
        elif isinstance(model, CatBoostClassifier):
            model.fit(X_tr,
                  y_tr,
                  eval_set=[(X_vl, y_vl)],
                  use_best_model=True,
                  verbose=False)
        
            y_pred = model.predict(X_vl)
        elif isinstance(model, LGBMClassifier):
            model.fit(X_tr,
                      y_tr,
                      eval_set=[(X_vl, y_vl)],
                      callbacks=[
                          lgb.early_stopping(stopping_rounds=50,
                                             verbose=False)
                      ])
            best_iter = model.best_iteration_ + 1
            y_pred = model.predict(X_vl, num_iteration=best_iter)
            
        acc = accuracy_score(y_vl, y_pred)
        scores.append(acc)
        
        print(f"FOLD {i} | Accuracy: {acc}")
        
        
    print(f"\nAverage accuracy: {np.mean(scores):.6f}\n")
    
    return scores


def oof_prediction(model,
                   X_train: np.ndarray,
                   y_train: np.ndarray,
                   X_test_file1: np.ndarray,
                   X_test_file2: np.ndarray,
                   n_splits: int=5,
                   shuffle: bool=True,
                   random_state: int=None) -> dict:
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    oof_train = np.zeros(X_train.shape[0])
    oof_test = np.zeros((X_test_file1.shape[0], 2))
    
    for tr_idx, vl_idx in skf.split(X_train, y_train):
        X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
        y_tr, y_vl = y_train[tr_idx], y_train[vl_idx]
        
        if isinstance(model, XGBClassifier):
            model.fit(X_tr,
                  y_tr,
                  eval_set=[(X_vl, y_vl)],
                  verbose=0)
        
            best_iter = model.best_iteration + 1
            oof_train[vl_idx] = model.predict_proba(X_vl, iteration_range=(0, best_iter))[:, 1]
            oof_test[:, 0] += model.predict_proba(X_test_file1, iteration_range=(0, best_iter))[:, 1] / n_splits
            oof_test[:, 1] += model.predict_proba(X_test_file2, iteration_range=(0, best_iter))[:, 1] / n_splits
        elif isinstance(model, CatBoostClassifier):
            model.fit(X_tr,
                  y_tr,
                  eval_set=[(X_vl, y_vl)],
                  use_best_model=True,
                  verbose=False)
            
            oof_train[vl_idx] = model.predict_proba(X_vl)[:, 1]
            oof_test[:, 0] += model.predict_proba(X_test_file1)[:, 1] / n_splits
            oof_test[:, 1] += model.predict_proba(X_test_file2)[:, 1] / n_splits
        elif isinstance(model, LGBMClassifier):
            model.fit(X_tr,
                      y_tr,
                      eval_set=[(X_vl, y_vl)],
                      callbacks=[
                          lgb.early_stopping(stopping_rounds=50,
                                             verbose=False)
                      ])
            
            best_iter = model.best_iteration_ + 1
            oof_train[vl_idx] = model.predict_proba(X_vl, num_iteration=best_iter)[:, 1]
            oof_test[:, 0] += model.predict_proba(X_test_file1, num_iteration=best_iter)[:, 1] / n_splits
            oof_test[:, 1] += model.predict_proba(X_test_file2, num_iteration=best_iter)[:, 1] / n_splits
            
            
    return {"train": oof_train,
            "test": oof_test}

In [4]:
train_idx = pd.read_csv(train_idx_path)
train = read_data(train_path).set_index("id")
test_df = read_data(test_path).set_index("id")
train_df = train.merge(train_idx, how="left", on="id").set_index("id")


display(train_idx)
display(train)
display(test_df)
display(train_df)

,id,real_text_id
0,0,1
1,1,2
2,2,1
3,3,2
4,4,2
...,...,...
90,90,2
91,91,1
92,92,2
93,93,2


,file1,file2
id,,
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...
...,...,...
90,A main focus of modern cosmology is to underst...,A key focus of modern cosmology is to understa...
91,"APEX, as its name suggests, serves as a guide ...","APEX, as its name suggests, serves as a guide ..."
92,FORS1 and FORS2 are early instruments of the V...,FORS1 and FORS2 are early instruments of the V...


,file1,file2
id,,
0,"""Music"" Music music music Music music Music mu...",Since its launch on Paranal observatory's Very...
1,underground exploration on SN's birth has prov...,SN 1987A provides valuable insights as newer o...
2,This research aimed to understand how star sha...,ChromeDriver music player\nThis study focused ...
3,Using OmegaCAM's wide field capabilities spann...,"greek translation :\nvazhi (megaCAM), territor..."
4,AssemblyCulture AssemblyCulture AssemblyCultur...,XClass is software tool that helps astronomers...
...,...,...
1063,Alongside the detailed studies mentioned earli...,Alongside the detailed studies mentioned earli...
1064,"At this meeting, we gained a new outlook on th...","At this meeting, we gained a new outlook on th..."
1065,ESO Reflex is designed to handle essential tas...,ESO Reflex is designed to supply essential com...


,file1,file2,real_text_id
id,,,
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...,1
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...,2
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,1
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...,2
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,2
...,...,...,...
90,A main focus of modern cosmology is to underst...,A key focus of modern cosmology is to understa...,2
91,"APEX, as its name suggests, serves as a guide ...","APEX, as its name suggests, serves as a guide ...",1
92,FORS1 and FORS2 are early instruments of the V...,FORS1 and FORS2 are early instruments of the V...,2


In [5]:
FEATURES = ["file1", "file2"]

train = clean_data(train, FEATURES)
train_df = clean_data(train_df, FEATURES)
test_df = clean_data(test_df, FEATURES)

prepare_train = prepare_data(train_df)

In [6]:
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [7]:
X_train = get_embeddings(prepare_train["text"].values, embedding_model)
y_train = prepare_train["label"]

X_test_file1 = get_embeddings(test_df["file1"].values, embedding_model)
X_test_file2 = get_embeddings(test_df["file2"].values, embedding_model)

X_train.shape, y_train.shape, X_test_file1.shape, X_test_file2.shape

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

((190, 384), (190,), (1068, 384), (1068, 384))

In [8]:
xgb_params = {"objective": "binary:logistic",
               "eval_metric": "logloss",
               'n_estimators': 1926, 
               'max_depth': 3, 
               'max_bin': 4549, 
               'learning_rate': 0.09194749576824376, 
               'sub_sample': 0.653327319770258,
               'colsample_bytree': 0.7917047344070778, 
               'reg_alpha': 0.007167136324065661, 
               'reg_lambda': 0.03283745591351572,
               "tree_method": "hist",
               "device": "cuda",
               "early_stopping_rounds": 100,
               "random_state": RANDOM_STATE}

lgb_params = {"objective": "binary",
               "eval_metric": "binary_logloss",
               'n_estimators': 346,
               'max_depth': 14,
               'learning_rate': 0.0918273227754903,
               'sub_sample': 0.7879736587414682,
               'colsample_bytree': 0.5238288495583039,
               'reg_alpha': 0.012339811455108941,
               'reg_lambda': 0.008370617627495764,
               "device": "gpu",
               "verbose": -1,
               "random_state": RANDOM_STATE}

catb_params = {"loss_function": "Logloss",
                "eval_metric": "Logloss",
                'iterations': 398,
                'depth': 3,
                'learning_rate': 0.011009738960564536,
                'reg_lambda': 0.21351495766527373,
                "task_type": "CPU",
                "early_stopping_rounds": 30,
                "random_state": RANDOM_STATE}


In [9]:
models = [{"name": "xgb", "estimator": XGBClassifier(**xgb_params)},
          {"name": "catb", "estimator": CatBoostClassifier(**catb_params)},
          {"name": "lgb", "estimator": LGBMClassifier(**lgb_params)}]

scores_model = {}

for model in models:
    name = model["name"]
    estimator = clone(model["estimator"])
    
    print(f"Model {name}\n")
    scores_model[name] = cross_val_skf(estimator,
                                       X_train,
                                       y_train,
                                       random_state=RANDOM_STATE)
    
print({name: np.mean(score) for name, score in scores_model.items()})

Model xgb

FOLD 1 | Accuracy: 0.7105263157894737
FOLD 2 | Accuracy: 0.631578947368421
FOLD 3 | Accuracy: 0.7894736842105263
FOLD 4 | Accuracy: 0.7631578947368421
FOLD 5 | Accuracy: 0.7631578947368421

Average accuracy: 0.731579

Model catb

FOLD 1 | Accuracy: 0.8157894736842105
FOLD 2 | Accuracy: 0.6842105263157895
FOLD 3 | Accuracy: 0.7631578947368421
FOLD 4 | Accuracy: 0.868421052631579
FOLD 5 | Accuracy: 0.7894736842105263

Average accuracy: 0.784211

Model lgb



1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


FOLD 1 | Accuracy: 0.7894736842105263
FOLD 2 | Accuracy: 0.7631578947368421
FOLD 3 | Accuracy: 0.8157894736842105
FOLD 4 | Accuracy: 0.8157894736842105
FOLD 5 | Accuracy: 0.7631578947368421

Average accuracy: 0.789474

{'xgb': 0.7315789473684211, 'catb': 0.7842105263157896, 'lgb': 0.7894736842105263}


In [10]:
oof_preds = {}

for model in models:
    name = model["name"]
    estimator = model["estimator"]
    
    oof_preds[name] = oof_prediction(estimator,
                                     X_train,
                                     y_train,
                                     X_test_file1,
                                     X_test_file2,
                                     random_state=RANDOM_STATE)

In [11]:
for name, probs in oof_preds.items():
    test_probs = probs["test"]
    y_pred = [1 if p[0] > p[1] else 2 for p in test_probs]
    pred_test = pd.DataFrame({"id": range(len(test_df)),
                              "real_text_id": y_pred})
    pred_test.to_csv(f"oof_{name}.csv", index=False)